# 🔁 Défi quotidien : Le reclassement sans serveur Pinecone en action

Ce notebook implémente un pipeline de reclassement (reranking) avec Pinecone, en deux parties :
- **Partie 1** : Reclassement basique de documents
- **Parties 2–6** : Index sans serveur avec des notes médicales

---
## 🧩 Partie 1 : Charger les documents et exécuter le modèle de reclassement

In [ ]:
# Étape 1.1 — Installer les bibliothèques Pinecone
!pip install -U pinecone==6.0.1 pinecone-notebooks

In [ ]:
# Étape 1.2 — Authentification avec Pinecone
import os

# Clé API définie directement (à ne jamais partager publiquement)
os.environ["PINECONE_API_KEY"] = "pcsk_5MYS2R_JmeCWrqK6HuM3DBu5trn2boBpy7vFLmSMcJAFVYgRTYn1r3CYztHfyeC3hBVtAX"

if not os.environ.get("PINECONE_API_KEY"):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

print("✅ Clé API définie avec succès.")

In [ ]:
# Étape 1.3 — Instancier le client Pinecone
from pinecone import Pinecone

api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=api_key)

print("✅ Client Pinecone initialisé.")

In [ ]:
# Étape 1.4 — Définir la requête et les documents
query = "Tell me about Apple's products"

documents = [
    "The apple is a sweet, edible fruit produced by the apple tree. It is rich in fiber and vitamins.",
    "Apple Inc. is a technology company known for its iPhone, iPad, MacBook, and Apple Watch products.",
    "Apples come in many varieties such as Granny Smith, Fuji, and Gala. They are widely consumed worldwide.",
    "Apple's latest iPhone model features an advanced A17 chip, improved camera system, and longer battery life.",
    "Eating an apple a day is a popular saying; apples contain antioxidants and are good for heart health."
]

print(f"✅ {len(documents)} documents définis.")

In [ ]:
# Étape 1.5 — Appel au service de reclassement
from pinecone import RerankModel

reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3  # On récupère les 3 documents les plus pertinents
)

print("✅ Reclassement effectué.")

In [ ]:
# Étape 1.6 — Afficher les résultats reclassés
def show_reranked_results(query, matches):
    print(f"Query: {query}\n")
    for i, m in enumerate(matches):
        print(f"  {i+1}. Score: {m.score:.4f}")
        print(f"     Texte : {m.document.text}\n")

show_reranked_results(query, reranked.data)

---
## 🏥 Partie 2 : Mise en place d'un index sans serveur pour les notes médicales

In [ ]:
# Étape 2.1 — Installer les bibliothèques de données et de modèles
!pip install pandas torch transformers

In [ ]:
# Étape 2.2 — Importer les modules et définir les paramètres d'environnement
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Paramètres cloud (valeurs par défaut pour la plupart des utilisateurs)
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Spécifications sans serveur
spec = ServerlessSpec(cloud=cloud, region=region)

# Nom de l'index
index_name = 'medical-notes-index'

print(f"✅ Configuration : cloud={cloud}, region={region}, index={index_name}")

In [ ]:
# Étape 2.3 — Créer ou recréer l'index

# Supprimer l'index existant s'il existe
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
    print(f"🗑️ Index '{index_name}' supprimé.")

# Créer un nouvel index
pc.create_index(
    name=index_name,
    dimension=384,       # Correspond au modèle all-MiniLM-L6-v2
    metric='cosine',     # Similarité cosinus recommandée pour le texte
    spec=spec
)

print(f"✅ Index '{index_name}' créé avec dimension=384 et metric='cosine'.")

---
## 📂 Partie 3 : Charger les données d'exemple

In [ ]:
# Étape 3.1 — Télécharger et lire le fichier JSONL
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    url = "https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

print("✅ Données téléchargées avec succès.")

In [ ]:
# Étape 3.2 — Prévisualiser le DataFrame
print("Data shape:", df.shape)  # (lignes, colonnes)
df.head()

---
## ⬆️ Partie 4 : Mise à jour des données dans l'index

In [ ]:
# Étape 4.1 — Instancier le client d'index et effectuer une mise à jour
index = pc.Index(name=index_name)

# Insérer les données depuis le DataFrame
index.upsert_from_dataframe(df)

print("✅ Données insérées dans l'index.")

In [ ]:
# Étape 4.2 — Attendre la disponibilité de l'index
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: {vector_count}")
    return vector_count > 0  # Attendre qu'au moins 1 vecteur soit indexé

while not is_fresh(index):
    time.sleep(5)

print("✅ Index prêt !")
index.describe_index_stats()

---
## 🔍 Partie 5 : Fonction de requête et d'intégration

In [ ]:
# Étape 5.1 — Définir la fonction d'intégration
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
    # Moyenne sur la dimension 0 (longueur de séquence) → vecteur unique par entrée
    embedding = model_output.last_hidden_state[0].mean(dim=0)
    return embedding

print("✅ Fonction d'intégration définie.")

In [ ]:
# Étape 5.2 — Exécuter une requête de recherche sémantique
question = "patient with chest pain"
query = get_embedding(question).tolist()

# Obtenir les résultats
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Trier par score décroissant
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

print(f"✅ {len(sorted_matches)} résultats obtenus.")

---
## 📊 Partie 6 : Afficher et réorganiser les notes cliniques

In [ ]:
# Étape 6.1 — Afficher les premiers résultats de la recherche
def show_results(question, matches):
    print(f"Question: '{question}'")
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f'     Score: {match["score"]}')
        print(f'     Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

In [ ]:
# Étape 6.2 — Préparer les documents pour le reclassement
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

print("✅ Documents transformés pour le reclassement.")
print("Exemple :", transformed_documents[0])

In [ ]:
# Étape 6.3 — Exécuter le reclassement sans serveur
refined_query = "patient with acute chest pain requiring urgent cardiac evaluation"

reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,            # Top 3 résultats reclassés
    return_documents=True,
)

print("✅ Reclassement des notes médicales effectué.")

In [ ]:
# Étape 6.4 — Afficher les résultats reclassés
def show_reranked_results(question, matches):
    print(f"Question: '{question}'")
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f'     Score: {match.score}')
        print(f'     Reranking Field: {match.document.reranking_field}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

In [ ]:
# Étape 6.5 — Nettoyage (facultatif)
# Décommentez la ligne ci-dessous pour supprimer l'index et éviter des frais inutiles
# pc.delete_index(name=index_name)
# print(f"🗑️ Index '{index_name}' supprimé.")

---
## 🎯 Critères de réussite

- ✅ Authentification réussie auprès de Pinecone
- ✅ Reclassement basique et visualisation des résultats pertinents
- ✅ Création et alimentation d'un index sans serveur avec des notes médicales
- ✅ Requêtes de recherche sémantique sur des données médicales
- ✅ Comparaison des résultats initiaux avec les résultats reclassés
- ✅ Compréhension de l'amélioration de la pertinence via le reclassement